# 01 Dataset audit — fixed

Fixed version. Main changes:
- Removed the hard-coded local Kaggle cache path.
- Creates `splits/`, `checkpoints/`, and `results/` folders automatically.
- Keeps the patient-level split and dataset audit outputs.

In [1]:
import os
import random
from pathlib import Path

import kagglehub
import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Create output folders used by later notebooks.
for folder in ["splits", "checkpoints", "results"]:
    os.makedirs(folder, exist_ok=True)

# Download / locate the NIH ChestX-ray14 dataset through kagglehub.
# This is portable: it works on your Mac and on another machine without hard-coding /Users/shuu/...
dataset_path = kagglehub.dataset_download("nih-chest-xrays/data")
dataset_path = str(Path(dataset_path))

print("Path to dataset files:", dataset_path)
print("Top-level files/folders:", os.listdir(dataset_path)[:20])

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: /Users/shuu/.cache/kagglehub/datasets/nih-chest-xrays/data/versions/3
Top-level files/folders: ['images_006', 'images_001', 'images_008', 'images_009', 'images_007', 'FAQ_CHESTXRAY.pdf', 'images_012', 'Data_Entry_2017.csv', 'BBox_List_2017.csv', 'ARXIV_V5_CHESTXRAY.pdf', 'train_val_list.txt', 'README_CHESTXRAY.pdf', 'images_002', 'images_005', 'LOG_CHESTXRAY.pdf', 'images_004', 'images_003', 'test_list.txt', 'images_010', 'images_011']


In [2]:
# Dataset path now comes directly from kagglehub, not from a hard-coded local path.
print("Dataset path:", dataset_path)
print("Output folders ready:", [folder for folder in ["splits", "checkpoints", "results"] if os.path.exists(folder)])

Dataset path: /Users/shuu/.cache/kagglehub/datasets/nih-chest-xrays/data/versions/3
Output folders ready: ['splits', 'checkpoints', 'results']


In [3]:
csv_path = os.path.join(dataset_path, "Data_Entry_2017.csv")

df = pd.read_csv(csv_path)

print(df.shape)
print(df.columns)
df.head()

(112120, 12)
Index(['Image Index', 'Finding Labels', 'Follow-up #', 'Patient ID',
       'Patient Age', 'Patient Gender', 'View Position', 'OriginalImage[Width',
       'Height]', 'OriginalImagePixelSpacing[x', 'y]', 'Unnamed: 11'],
      dtype='object')


,Image Index,Finding Labels,Follow-up #,Patient ID,Patient Age,Patient Gender,View Position,OriginalImage[Width,Height],OriginalImagePixelSpacing[x,y],Unnamed: 11
0,00000001_000.png,Cardiomegaly,0,1,58,M,PA,2682,2749,0.143,0.143,NaN
1,00000001_001.png,Cardiomegaly|Emphysema,1,1,58,M,PA,2894,2729,0.143,0.143,NaN
2,00000001_002.png,Cardiomegaly|Effusion,2,1,58,M,PA,2500,2048,0.168,0.168,NaN
3,00000002_000.png,No Finding,0,2,81,M,PA,2500,2048,0.171,0.171,NaN
4,00000003_000.png,Hernia,0,3,81,F,PA,2582,2991,0.143,0.143,NaN


In [4]:
from glob import glob
import os

image_paths = glob(os.path.join(dataset_path, "**", "*.png"), recursive=True)

print("Number of images:", len(image_paths))
print(image_paths[:5])

Number of images: 112120
['/Users/shuu/.cache/kagglehub/datasets/nih-chest-xrays/data/versions/3/images_006/images/00011723_001.png', '/Users/shuu/.cache/kagglehub/datasets/nih-chest-xrays/data/versions/3/images_006/images/00013648_003.png', '/Users/shuu/.cache/kagglehub/datasets/nih-chest-xrays/data/versions/3/images_006/images/00011945_002.png', '/Users/shuu/.cache/kagglehub/datasets/nih-chest-xrays/data/versions/3/images_006/images/00012342_003.png', '/Users/shuu/.cache/kagglehub/datasets/nih-chest-xrays/data/versions/3/images_006/images/00011955_002.png']


In [5]:
print("Gender counts:")
print(df["Patient Gender"].value_counts())

print("\nView position counts:")
print(df["View Position"].value_counts())

labels = df["Finding Labels"].str.split("|").explode()
print("\nDisease label counts:")
print(labels.value_counts())

Gender counts:
Patient Gender
M    63340
F    48780
Name: count, dtype: int64

View position counts:
View Position
PA    67310
AP    44810
Name: count, dtype: int64

Disease label counts:
Finding Labels
No Finding            60361
Infiltration          19894
Effusion              13317
Atelectasis           11559
Nodule                 6331
Mass                   5782
Pneumothorax           5302
Consolidation          4667
Pleural_Thickening     3385
Cardiomegaly           2776
Emphysema              2516
Edema                  2303
Fibrosis               1686
Pneumonia              1431
Hernia                  227
Name: count, dtype: int64


In [6]:
df["Effusion_label"] = df["Finding Labels"].str.contains("Effusion", regex=False).astype(int)

print("Effusion label counts:")
print(df["Effusion_label"].value_counts())

print("\nEffusion by gender:")
print(pd.crosstab(df["Patient Gender"], df["Effusion_label"], margins=True))

Effusion label counts:
Effusion_label
0    98803
1    13317
Name: count, dtype: int64

Effusion by gender:
Effusion_label      0      1     All
Patient Gender                      
F               42898   5882   48780
M               55905   7435   63340
All             98803  13317  112120


In [7]:
# Define target and protected attribute
target_label = "Effusion"

df["target"] = df["Finding Labels"].str.contains(target_label, regex=False).astype(int)
df["sex"] = df["Patient Gender"].map({"F": 0, "M": 1})

print(df[["Image Index", "Finding Labels", "Patient ID", "Patient Gender", "target", "sex"]].head())
print(df["target"].value_counts())
print(df["sex"].value_counts())

        Image Index          Finding Labels  Patient ID Patient Gender  \
0  00000001_000.png            Cardiomegaly           1              M   
1  00000001_001.png  Cardiomegaly|Emphysema           1              M   
2  00000001_002.png   Cardiomegaly|Effusion           1              M   
3  00000002_000.png              No Finding           2              M   
4  00000003_000.png                  Hernia           3              F   

   target  sex  
0       0    1  
1       0    1  
2       1    1  
3       0    1  
4       0    0  
target
0    98803
1    13317
Name: count, dtype: int64
sex
1    63340
0    48780
Name: count, dtype: int64


In [8]:
from glob import glob
import os

image_paths = glob(os.path.join(dataset_path, "**", "*.png"), recursive=True)

image_path_dict = {
    os.path.basename(path): path
    for path in image_paths
}

df["image_path"] = df["Image Index"].map(image_path_dict)

print("Number of images:", len(image_paths))
print("Missing image paths:", df["image_path"].isna().sum())
df[["Image Index", "image_path"]].head()

Number of images: 112120
Missing image paths: 0


,Image Index,image_path
0,00000001_000.png,/Users/shuu/.cache/kagglehub/datasets/nih-ches...
1,00000001_001.png,/Users/shuu/.cache/kagglehub/datasets/nih-ches...
2,00000001_002.png,/Users/shuu/.cache/kagglehub/datasets/nih-ches...
3,00000002_000.png,/Users/shuu/.cache/kagglehub/datasets/nih-ches...
4,00000003_000.png,/Users/shuu/.cache/kagglehub/datasets/nih-ches...


In [9]:
print("Missing target:", df["target"].isna().sum())
print("Missing sex:", df["sex"].isna().sum())
print("Missing Patient ID:", df["Patient ID"].isna().sum())
print("Missing image path:", df["image_path"].isna().sum())

Missing target: 0
Missing sex: 0
Missing Patient ID: 0
Missing image path: 0


In [10]:
from sklearn.model_selection import GroupShuffleSplit

# Use Patient ID as group to avoid patient leakage
groups = df["Patient ID"]

# 1) Split into train+val and test
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_val_idx, test_idx = next(
    gss.split(df, y=df["target"], groups=groups)
)

train_val_df = df.iloc[train_val_idx].reset_index(drop=True)
test_df = df.iloc[test_idx].reset_index(drop=True)

# 2) Split train+val into train and validation
gss_val = GroupShuffleSplit(
    n_splits=1,
    test_size=0.125,  # 0.125 of 80% = 10% of full dataset
    random_state=42
)

train_idx, val_idx = next(
    gss_val.split(
        train_val_df,
        y=train_val_df["target"],
        groups=train_val_df["Patient ID"]
    )
)

train_df = train_val_df.iloc[train_idx].reset_index(drop=True)
val_df = train_val_df.iloc[val_idx].reset_index(drop=True)

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Train: (78873, 16)
Validation: (10953, 16)
Test: (22294, 16)


In [11]:
train_patients = set(train_df["Patient ID"])
val_patients = set(val_df["Patient ID"])
test_patients = set(test_df["Patient ID"])

print("Train-Val overlap:", len(train_patients & val_patients))
print("Train-Test overlap:", len(train_patients & test_patients))
print("Val-Test overlap:", len(val_patients & test_patients))

Train-Val overlap: 0
Train-Test overlap: 0
Val-Test overlap: 0


In [12]:
def audit_split(name, split_df):
    print(f"\n===== {name} =====")
    print("Images:", len(split_df))
    print("Patients:", split_df["Patient ID"].nunique())

    print("\nTarget distribution:")
    print(split_df["target"].value_counts())
    print(split_df["target"].value_counts(normalize=True))

    print("\nGender distribution:")
    print(split_df["Patient Gender"].value_counts())
    print(split_df["Patient Gender"].value_counts(normalize=True))

    print("\nEffusion by gender:")
    print(pd.crosstab(split_df["Patient Gender"], split_df["target"], margins=True))

audit_split("Train", train_df)
audit_split("Validation", val_df)
audit_split("Test", test_df)


===== Train =====
Images: 78873
Patients: 21563

Target distribution:
target
0    69340
1     9533
Name: count, dtype: int64
target
0    0.879135
1    0.120865
Name: proportion, dtype: float64

Gender distribution:
Patient Gender
M    44437
F    34436
Name: count, dtype: int64
Patient Gender
M    0.563399
F    0.436601
Name: proportion, dtype: float64

Effusion by gender:
target              0     1    All
Patient Gender                    
F               30216  4220  34436
M               39124  5313  44437
All             69340  9533  78873

===== Validation =====
Images: 10953
Patients: 3081

Target distribution:
target
0    9722
1    1231
Name: count, dtype: int64
target
0    0.887611
1    0.112389
Name: proportion, dtype: float64

Gender distribution:
Patient Gender
M    6162
F    4791
Name: count, dtype: int64
Patient Gender
M    0.562586
F    0.437414
Name: proportion, dtype: float64

Effusion by gender:
target             0     1    All
Patient Gender                   
F    

In [13]:
os.makedirs("splits", exist_ok=True)

split_cols = [
    "Image Index",
    "image_path",
    "Finding Labels",
    "Patient ID",
    "Patient Age",
    "Patient Gender",
    "View Position",
    "target",
    "sex"
]

train_df[split_cols].to_csv("splits/train.csv", index=False)
val_df[split_cols].to_csv("splits/val.csv", index=False)
test_df[split_cols].to_csv("splits/test.csv", index=False)

print("Saved:")
print("splits/train.csv")
print("splits/val.csv")
print("splits/test.csv")

Saved:
splits/train.csv
splits/val.csv
splits/test.csv


In [14]:
import os

print(os.listdir("splits"))

['val.csv', 'test.csv', 'train.csv']
